# Honderd epochs: een gewichtsmodus, met afzonderlijke controle

CNN-8 / SVHN train-only. MXFP6 E2M3/A32 of W32/A32; FP32-updatemasters en Adam.
Dit notebook leest alleen opgeslagen JSON/CSV. Geen training, modelcalls, downloads of wijzigingen aan ruwe gegevens.
De controle is een **afzonderlijke training**, geen tweede modus in hetzelfde model. Een seed per modus is geen statistische gelijkwaardigheidsstudie.


In [ ]:
from pathlib import Path
RUN_DIR = Path.cwd()  # Automatisch ingevuld in de notebookkopie in de runmap.
CONTROL_DIR = None  # Optioneel vanuit de MX-run: Path(r'C:\...\..._control_run')


In [ ]:
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

required = ['run_report.json', 'epoch_metrics.csv', 'selected_metrics.csv', 'history.json',
            'epoch_times.csv', 'weight_updates.csv', 'training_summary.json']
missing = [name for name in required if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Geen volledige trainingsrapportage in {RUN_DIR}: {missing}")
def read_json(name, root=RUN_DIR):
    return json.loads((root / name).read_text(encoding='utf-8'))
report = read_json('run_report.json')
history = read_json('history.json')
config = report['config']
if config.get('schema') != 'mx-weights-single-reference100-protocol-v1':
    raise ValueError('Verkeerd protocol: gebruik de honderd-epochrun, niet de pilot.')
metrics = pd.read_csv(RUN_DIR / 'epoch_metrics.csv')
selected = pd.read_csv(RUN_DIR / 'selected_metrics.csv')
times = pd.read_csv(RUN_DIR / 'epoch_times.csv')
updates = pd.read_csv(RUN_DIR / 'weight_updates.csv')
summary = read_json('training_summary.json')
records = history['epochs']
synthetic = bool(report.get('synthetic_fixture_only', False))
if synthetic:
    display(Markdown('**SYNTHE TISCHE SOFTWARETEST: geen SVHN-resultaten of GPU-tijdmeting.**'.replace('SYNTHE TISCHE','SYNTHETISCHE')))
if report.get('status') != 'PASS' or not summary['full_100_epoch_training_completed']:
    display(Markdown('**Onvoltooide of niet geslaagde run. Niet als honderd-epochresultaat rapporteren.**'))
display(pd.DataFrame([{'status': report['status'], 'mode': config['mode'],
    'gewicht': config['weight_format'], 'activaties': config['activation_bits'],
    'master': config['master_dtype'], 'epochs': report['completed_epoch'],
    'updates': report['optimizer_steps'], 'distillatie': config['distillation'],
    'truncatie': config['truncation'], 'volledige_training': summary['full_100_epoch_training_completed']}]))
display(selected[['role','epoch','weight_format','n','correct1','count_top1_percent','objective_loss']])


## Integriteit en checkpointselectie

Beste = het eerste strikte maximum van testaccuracy in de actieve modus. Laatste = epoch 100 bij een complete run.
De testset is daardoor **niet onafhankelijk van checkpointselectie**. Voor de hoofdvergelijking tussen de twee runs gebruiken we ook expliciet epoch 100 tegenover epoch 100.
Alle onderstaande controles steunen op rapporten; dit notebook herhaalt geen modelevaluatie.


In [ ]:
expected_steps = math.ceil(config['train_examples'] / config['batch_size'])
def expected_lr(epoch, for_next=False):
    lr = config['lr']
    for milestone in config['scheduler_milestones']:
        if milestone < epoch or (for_next and milestone == epoch):
            lr *= config['scheduler_gamma']
    return lr
best = records[0]
for r in records[1:]:
    if r['test'][0]['top1_percent'] > best['test'][0]['top1_percent']:
        best = r
selection = read_json('checkpoint_selection.json')
checks = {
    'Epochs vormen een ononderbroken reeks': [r['epoch'] for r in records] == list(range(1, len(records)+1)),
    'Een train- en een testrij per epoch': len(metrics) == 2 * len(records),
    'Geen dubbele metriccombinaties': not metrics.duplicated(['epoch','phase']).any(),
    'Eindige loss en accuracies': all(math.isfinite(float(v)) for col in ['objective_loss','top1_percent'] for v in metrics[col]),
    'Correcte aantallen consistent': ((100*metrics.correct1/metrics.n - metrics.count_top1_percent).abs() < 1e-8).all(),
    'Datasetgroottes kloppen': all(row.n == config['train_examples' if row.phase == 'train' else 'test_examples'] for row in metrics.itertuples()),
    'LR gebruikt en volgend LR kloppen': all(r['lr_used'] == expected_lr(r['epoch']) and r['lr_next'] == expected_lr(r['epoch'], True) for r in records),
    'Alle optimizerupdates geteld': all(r['optimizer_steps'] == r['epoch']*expected_steps for r in records),
    'BN-tellers gecontroleerd': all(r['batchnorm_counters_correct'] for r in records),
    'Geen optimizerupdates bij test': all(r['test_optimizer_steps'] == 0 for r in records),
    'Alle gradients voor optimizerupdate gecontroleerd': all(r['gradient_checks']['finite_gradient_checks'] == expected_steps for r in records),
    'FP32-Adamstaat gecontroleerd': all(r['adam_checks']['moments_fp32_finite'] for r in records),
    'Beste selectie opnieuw bevestigd': best['epoch'] == selection['best']['epoch'],
    'Laatste selectie opnieuw bevestigd': records[-1]['epoch'] == selection['last']['epoch'],
    'Alleen complete honderd epochs als voltooid': summary['full_100_epoch_training_completed'] == (len(records) == 100),
}
display(pd.DataFrame([{'controle': k, 'geslaagd': bool(v)} for k,v in checks.items()]))
if not all(checks.values()):
    raise ValueError('Een rapportcontrole faalt: eerst onderzoeken, geen conclusie over MX trekken.')
display(pd.DataFrame(report.get('warnings', [])))


## Leer- en LR-verloop

Training gebruikt dropout en updates; evaluatie niet. Cross-entropy wordt voor beide fasen op echte labels berekend.
De LR daalt **na** epochs 50, 75 en 90. De eerste en laatste laag blijven FP32; alleen BN32 leert.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for phase, data in metrics.groupby('phase'):
    ax.plot(data.epoch, data.count_top1_percent, label=phase)
for epoch in config['scheduler_milestones']:
    ax.axvline(epoch + 0.5, linestyle=':', alpha=0.3)
ax.set(xlabel='Epoch', ylabel='Top-1 accuracy (%)', title=f"{config['weight_format']}/A32 - {'synthetisch' if synthetic else 'SVHN train-only'}")
ax.legend(); ax.grid(True, alpha=0.25); fig.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for phase, data in metrics.groupby('phase'):
    ax.plot(data.epoch, data.objective_loss, label=phase)
ax.set(xlabel='Epoch', ylabel='Hard-label cross-entropy', title='Loss - een forward/backward per minibatch')
ax.legend(); ax.grid(True, alpha=0.25); fig.tight_layout(); plt.show()
lr_table = pd.DataFrame([{'epoch':r['epoch'], 'lr_gebruikt':r['lr_used'], 'lr_volgende_epoch':r['lr_next']} for r in records])
display(lr_table[lr_table.epoch.isin([1, 50, 51, 75, 76, 90, 91, 100])])


## Gemeten tijd, apart van een extrapolatie

Na honderd epochs is de som van alle gemeten epochtijden leidend. De sessieduur bevat daarnaast onder meer opstarten en eindexport; bij hervatten betreft zij alleen de huidige sessie.
Onze software bevat codec- en controleoverhead. Dit is geen benchmark van native MX-hardware.


In [ ]:
estimate = read_json('timing_estimate.json')
measured = float(times.total_epoch_seconds.sum())
time_summary = {'epochs': len(times), 'gemeten_epochminuten': measured/60,
                'train_minuten':times.train_seconds.sum()/60, 'test_minuten':times.test_seconds.sum()/60,
                'checkpoint_minuten':times.checkpoint_seconds.sum()/60,
                'andere_epochcontroles_minuten':times.other_seconds.sum()/60,
                'huidige_sessie_minuten': report.get('session_wall_seconds', float('nan'))/60}
display(pd.DataFrame([time_summary]))
if synthetic:
    display(Markdown('**Synthetische CPU-tijden: geen productie- of GPU-tijdsraming.**'))
elif summary['full_100_epoch_training_completed']:
    display(Markdown(f"**Werkelijk gemeten honderd epochs: {measured/60:.2f} minuten.** Geen extrapolatie."))
elif estimate.get('available'):
    display(Markdown(f"Nog onvoltooid. **Extrapolatie:** {estimate['estimated_100_epoch_seconds']/60:.1f} minuten; geen gemeten volledige training."))
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(times.epoch, times.total_epoch_seconds)
ax.set(xlabel='Epoch', ylabel='Seconden', title='Totale gemeten epochtijd')
ax.grid(True, alpha=0.25); fig.tight_layout(); plt.show()


## Updates en MX-export

De FP32-masterwaarden worden aangepast. In de MX-run worden de MX-codes opnieuw uit actuele effectieve gewichten berekend.
MX-codes/schalen in de inferentie-export zijn **geen** vervanging voor het FP32-trainingscheckpoint met Adamstaat.
De exportcontrole controleert exacte gewichtstensors, niet opnieuw alle voorspellingen op de volledige testset.


In [ ]:
display(updates.groupby('layer')[['master_changed_values','master_update_l2']].agg(['min','max']))
if config['mode'] == 'mx':
    display(updates.groupby('layer')[['changed_element_codes_including_padding','changed_scale_codes']].agg(['min','max']))
    display(pd.DataFrame(report.get('mx_exports', {})).T)
check_epochs = [1, 5, 50, 51, 75, 76, 90, 91, 100]
display(updates[updates.epoch.isin(check_epochs)])
gradients = pd.DataFrame([r['gradient_checks'] | {'epoch':r['epoch']} for r in records])
display(gradients[gradients.epoch.isin(check_epochs)])


## Twee afzonderlijke runs vergelijken

Open dit notebook vanuit de **MX-run**, vul `CONTROL_DIR` in met de map van de volledige **control-run**, en voer opnieuw uit.
Het notebook weigert een vergelijking wanneer protocollen, initialisatie, alle train/test-volgordehashes, data of code niet overeenkomen.
Het vergelijkt gelijke epochs en toont de afzonderlijk geselecteerde beste resultaten apart. Een verschil is geen bewijs van statistische robuustheid; CUDA is niet bitexact deterministisch ingesteld.


In [ ]:
pair = None
if CONTROL_DIR is None:
    display(Markdown('Geen controlepad opgegeven; alleen de huidige run wordt getoond.'))
else:
    cp = Path(CONTROL_DIR)
    cr = read_json('run_report.json', cp)
    ch = read_json('history.json', cp)
    cc = cr['config']
    if config['mode'] != 'mx' or cc['mode'] != 'control':
        raise ValueError('Vergelijk vanuit de MX-run, met de control-run als CONTROL_DIR.')
    mode_fields = {'mode','weight_format','mx_block_size','mx_scale_format','mx_group_axis',
                   'mx_scale_policy','mx_rounding','ste_policy'}
    shared_config_equal = ({k:v for k,v in config.items() if k not in mode_fields} ==
                           {k:v for k,v in cc.items() if k not in mode_fields})
    epoch_ids_equal = [r['epoch'] for r in records] == [r['epoch'] for r in ch['epochs']]
    order_equal = epoch_ids_equal and all(x['coverage'][phase]['order_sha256'] == y['coverage'][phase]['order_sha256']
        for x,y in zip(records,ch['epochs']) for phase in ['train','test'])
    paired_checks = {
        'beide_geslaagd': report['status'] == cr['status'] == 'PASS',
        'beide_100_epochs': report['completed_epoch'] == cr['completed_epoch'] == 100,
        'protocol_gelijk_buiten_MX': shared_config_equal,
        'initialisatie_gelijk': report['initial_model_sha256'] == cr['initial_model_sha256'],
        'alle_train_test_volgordehashes_gelijk': order_equal,
        'omgeving_gelijk': report['environment'] == cr['environment'] and report['execution_settings'] == cr['execution_settings'],
        'databestanden_gelijk': report['dataset_files']['files'] == cr['dataset_files']['files'],
        'preprocessing_gelijk': report.get('transforms') == cr.get('transforms'),
        'code_gelijk': report['package_sha256'] == cr['package_sha256'],
        'gelijke_synthetische_status': synthetic == bool(cr.get('synthetic_fixture_only',False)),
    }
    display(pd.DataFrame([paired_checks]))
    if not all(paired_checks.values()):
        raise ValueError('Geen volledig gematchte honderd-epochvergelijking; onderzoek bovenstaande verschillen.')
    cm = pd.read_csv(cp/'epoch_metrics.csv')
    pair = metrics.query("phase == 'test'")[['epoch','count_top1_percent','objective_loss']].merge(
        cm.query("phase == 'test'")[['epoch','count_top1_percent','objective_loss']], on='epoch',
        suffixes=('_mx','_control'), validate='one_to_one')
    pair['mx_min_control_procentpunt'] = pair.count_top1_percent_mx - pair.count_top1_percent_control
    display(Markdown('### Vooraf vastgelegde eindvergelijking: epoch 100 tegenover epoch 100'))
    display(pair[pair.epoch == 100])
    display(Markdown('### Afzonderlijk geselecteerde beste en laatste checkpoints'))
    control_selected = pd.read_csv(cp/'selected_metrics.csv')
    display(pd.concat([selected.assign(experiment='MX'),control_selected.assign(experiment='controle')])[
        ['experiment','role','epoch','count_top1_percent','objective_loss','correct1','n']])


In [ ]:
if pair is not None:
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(pair.epoch, pair.count_top1_percent_control, label='W32/A32-controle')
    ax.plot(pair.epoch, pair.count_top1_percent_mx, label='MX E2M3/A32')
    ax.set(xlabel='Epoch', ylabel='Testaccuracy (%)', title='Gematchte eenprecisietrainingen')
    ax.legend(); ax.grid(True, alpha=0.25); fig.tight_layout(); plt.show()


In [ ]:
if pair is not None:
    fig, ax = plt.subplots(figsize=(8,4))
    ax.plot(pair.epoch, pair.mx_min_control_procentpunt)
    ax.axhline(0, linestyle=':', alpha=0.5)
    ax.set(xlabel='Epoch', ylabel='MX minus controle (procentpunt)', title='Verschil op dezelfde epochs')
    ax.grid(True, alpha=0.25); fig.tight_layout(); plt.show()
    display(pair[pair.epoch.isin([1,5,50,51,75,76,90,91,100])])
